# TXF Data Access Layer — Drive smoke test

Read-only smoke test for the recorded 185-day / 206,938-row TXF validation slice. It never downloads market data, rewrites cache files, infers a roll rule, or promotes all 473 weekly partitions.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import subprocess, sys
REPO = '/content/market-research-core'
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/hh4832/market-research-core.git', REPO], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO], check=True)
if REPO not in sys.path:
    sys.path.insert(0, REPO)


In [ ]:
from pathlib import Path

CACHE_DIR = Path('/content/drive/MyDrive/00Quant_Research/data/fubon_txf')
MANIFEST = Path('/content/drive/MyDrive/00Quant_Research/manifests/txf_access_manifest.json')

assert CACHE_DIR.is_dir(), f'TXF cache not found: {CACHE_DIR}'
parquets = list(CACHE_DIR.glob('*.parquet'))
sidecars = list(CACHE_DIR.glob('*.json'))
print('Raw cache inventory (not a GO verdict):', len(parquets), 'parquet,', len(sidecars), 'json')
assert len(parquets) == 473 and len(sidecars) == 473
assert MANIFEST.is_file(), (
    'STOPPED_FAIL_CLOSED: approved cache-to-validation manifest is missing. '
    'Do not substitute all 473 cache partitions.'
)


In [ ]:
from market_research_core import TXFAdapter

loaded = TXFAdapter.from_manifest(CACHE_DIR, MANIFEST).load()
print('TXF DAL smoke: PASS')
print('Verdict:', loaded.metadata.verdict)
print('Scope:', loaded.metadata.scope)
print('Rows:', len(loaded.frame))
print('Partitions:', len(loaded.metadata.source_files))
print('Exact duplicates removed:', loaded.metadata.exact_duplicates_removed)
